# Neural BP Polar Decoder — PBCH (N=512, K=56)
Original Polar-NN-MULT style, converted to TF2
- N=512, K=56 (K_info=32 + CRC-24), E=864, QPSK
- BP decoder with learnable edge weights LV / RV
- RNN weight sharing across BP iterations
- SGD optimizer (matches original)

In [1]:
import subprocess
result = subprocess.run(
    ['pip', 'install', 'torch', 'torchvision', 'torchaudio',
     '--index-url', 'https://download.pytorch.org/whl/cu121'],
    capture_output=True, text=True
)
print(result.stdout[-3000:])
print(result.stderr[-500:])

Looking in indexes: https://download.pytorch.org/whl/cu121




In [2]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

Wed Mar 25 19:24:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 581.29                 Driver Version: 581.29         CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4060 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   62C    P8              7W /   45W |       0MiB /   8188MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
import subprocess
result = subprocess.run(
    ['pip', 'install', 'torch', 'torchvision', 'torchaudio',
     '--index-url', 'https://download.pytorch.org/whl/cu121'],
    capture_output=True, text=True
)
print(result.stdout[-3000:])
print(result.stderr[-500:])

Looking in indexes: https://download.pytorch.org/whl/cu121




In [4]:
import torch
import torch.nn as nn
import numpy as np
import time
import os
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch : {torch.__version__}')
print(f'Device  : {device}')
print(f'GPU     : {torch.cuda.get_device_name(0)}')

PyTorch : 2.5.1+cu121
Device  : cuda
GPU     : NVIDIA GeForce RTX 4060 Laptop GPU


In [5]:
CHANNEL_TYPE = 'PBCH'
N       = 512
K_info  = 32
CRC_LEN = 24
K       = K_info + CRC_LEN   # 56
E       = 864
Rc      = K / E
Rm      = 2
n       = int(np.log2(N))

ebn0      = np.arange(-3, 7, 1, dtype=np.float32)
ebn0_num  = 10**(ebn0 / 10)
SNR       = ebn0 + 10*np.log10(Rc * Rm)
SNR_num   = 10**(SNR / 10).astype(np.float32)
noise_var = (1.0 / (2.0 * SNR_num)).astype(np.float32)

numOfWord     = 100
batch_size    = numOfWord * len(ebn0)
batches_train = 200
batches_val   = 40
batches_test  = 600
patience      = 10
bp_iter_num   = 5
RNN           = 1

print(f'N={N}  K={K}  K_info={K_info}  CRC={CRC_LEN}  E={E}')
print(f'Rc={Rc:.4f}  batch_size={batch_size}')
print(f'ebn0     : {ebn0}')
print(f'SNR      : {np.round(SNR,2)}')
print(f'noise_var: {np.round(noise_var,4)}')
print(f'BP iters : {bp_iter_num}  RNN: {RNN}')

N=512  K=56  K_info=32  CRC=24  E=864
Rc=0.0648  batch_size=1000
ebn0     : [-3. -2. -1.  0.  1.  2.  3.  4.  5.  6.]
SNR      : [-11.87 -10.87  -9.87  -8.87  -7.87  -6.87  -5.87  -4.87  -3.87  -2.87]
noise_var: [7.696  6.1132 4.8559 3.8571 3.0638 2.4337 1.9332 1.5356 1.2197 0.9689]
BP iters : 5  RNN: 1


In [6]:
CRC24C_POLY = np.array(
    [1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,1,1],
    dtype=int)

def crc_encode(info_bits):
    padded = np.concatenate([info_bits, np.zeros(CRC_LEN, dtype=int)])
    for i in range(len(info_bits)):
        if padded[i]:
            padded[i:i+25] ^= CRC24C_POLY
    return np.concatenate([info_bits, padded[-CRC_LEN:]])

def crc_check(bits):
    padded = bits.copy()
    for i in range(len(bits) - CRC_LEN):
        if padded[i]:
            padded[i:i+25] ^= CRC24C_POLY
    return np.all(padded[-CRC_LEN:] == 0)

_t = np.random.randint(0, 2, K_info)
_e = crc_encode(_t)
assert len(_e) == K and crc_check(_e), 'CRC self-check failed'
print(f'CRC-24C verified')
print(f'  poly length : {len(CRC24C_POLY)}')
print(f'  input length: {len(_t)}  output length: {len(_e)}')

CRC-24C verified
  poly length : 25
  input length: 32  output length: 56


In [7]:
# ============================================================
# Polar matrix + frozen bits from file
# ============================================================
Fi = np.ones([1])
for i in range(n):
    Fi = np.vstack((np.hstack((Fi, Fi*0)), np.hstack((Fi, Fi))))
F_kron_n = Fi.astype(np.float32)

bitreversedindices = np.zeros(N, dtype=int)
for i in range(N):
    b = '{:0{width}b}'.format(i, width=n)
    bitreversedindices[i] = int(b[::-1], 2)

indices  = np.loadtxt('FrozenBit/512.txt', dtype=int) - 1
FZlookup = np.zeros(N, dtype=np.float32)
FZlookup[indices[:K]] = -1
info_pos = np.where(FZlookup == -1)[0]

print(f'F_kron_n : {F_kron_n.shape}')
print(f'Info bits: {len(info_pos)}  Frozen: {N-len(info_pos)}')

F_kron_n : (512, 512)
Info bits: 56  Frozen: 456


In [8]:
# ============================================================
# gendata
# ============================================================
def gendata(const=0, flag=False):
    batch  = batch_size
    repeat = int(np.ceil(E / N))
    X = np.zeros((batch, N), dtype=np.float32)
    Y = np.zeros((batch, K), dtype=np.float32)

    for i in range(len(ebn0)):
        si = const if flag else i
        s2 = noise_var[si]

        U_info = np.random.randint(0, 2, (numOfWord, K_info))
        U_crc  = np.array([crc_encode(u) for u in U_info], dtype=np.float32)

        UP = np.tile(FZlookup, (numOfWord, 1)).copy()
        UP[UP == -1] = U_crc.flatten()
        UP = UP[:, bitreversedindices]
        x  = np.mod(UP @ F_kron_n, 2).astype(np.float32)

        x_rep = np.tile(x, (1, repeat))[:, :E]
        xp    = x_rep.reshape(numOfWord, E//2, 2)
        tx    = ((1 - 2*xp[:,:,0]) + 1j*(1 - 2*xp[:,:,1])) / np.sqrt(2)

        noise = (np.random.normal(0, np.sqrt(s2), tx.shape) +
                 1j*np.random.normal(0, np.sqrt(s2), tx.shape))
        rx    = tx + noise

        scale  = 2.0 / s2
        llr_E  = np.stack([scale * np.real(rx),
                           scale * np.imag(rx)],
                          axis=2).reshape(numOfWord, E).astype(np.float32)

        llr_N = np.zeros((numOfWord, N), dtype=np.float32)
        for k in range(repeat):
            st = k*N; en = min(st+N, E)
            llr_N[:, :en-st] += llr_E[:, st:en]

        X[i*numOfWord:(i+1)*numOfWord] = llr_N
        Y[i*numOfWord:(i+1)*numOfWord] = U_crc

    return X, Y

print('Verifying gendata...')
Xv, Yv = gendata(0, False)
print(f'  X={Xv.shape}  Y={Yv.shape}')
print(f'  CRC: {sum(crc_check(Yv[i].astype(int)) for i in range(20))}/20')

Verifying gendata...
  X=(1000, 512)  Y=(1000, 56)
  CRC: 20/20


In [9]:
# ============================================================
# Neural BP Decoder — fully vectorized, smooth f_func
# ============================================================
class NeuralBPDecoderFast(nn.Module):
    def __init__(self, N, K, n, FZlookup, info_pos,
                 bp_iter_num=5, rnn_share=True):
        super().__init__()
        self.N           = N
        self.K           = K
        self.n           = n
        self.bp_iter_num = bp_iter_num
        self.rnn_share   = rnn_share
        self.inf_num     = 10.0

        w_iters = 1 if rnn_share else bp_iter_num
        self.LV = nn.Parameter(torch.ones(n, N, w_iters))
        self.RV = nn.Parameter(torch.ones(n, N, w_iters))

        self._build_schedule(N, n, FZlookup, info_pos)

    def _build_schedule(self, N, n, FZlookup, info_pos):
        frozen = [j for j in range(N) if FZlookup[j] == 0]
        self.register_buffer('frozen_pos',
            torch.tensor(frozen, dtype=torch.long))
        self.register_buffer('info_pos_t',
            torch.tensor(info_pos, dtype=torch.long))

        # R2L schedule
        self.r2l_ops = []
        for i in range(n, 0, -1):
            stage = n+1-i
            wi    = n-i
            a_list=[]; b_list=[]; r0_list=[]; r1_list=[]
            for phi in range(2**i):
                psi = int(np.floor(phi/2))
                if phi % 2 != 0:
                    for omega in range(2**(n-i)):
                        a_list.append(psi + 2*omega*(2**(i-1)))
                        b_list.append(psi + (2*omega+1)*(2**(i-1)))
                        r0_list.append(phi - 1 + omega*(2**i))
                        r1_list.append(phi + omega*(2**i))
            self.r2l_ops.append({
                'stage': stage, 'wi': wi,
                'a' : torch.tensor(a_list,  dtype=torch.long),
                'b' : torch.tensor(b_list,  dtype=torch.long),
                'r0': torch.tensor(r0_list, dtype=torch.long),
                'r1': torch.tensor(r1_list, dtype=torch.long),
            })

        # L2R schedule — even phi first then odd phi per stage
        self.l2r_ops = []
        for i in range(1, n+1):
            stage = n+1-i
            wi    = n-i
            a_e=[]; b_e=[]; r1_e=[]; r1_ep1=[]
            a_o=[]; b_o=[]; r1_o=[]; r1_om1=[]
            for phi in range(2**i):
                psi = int(np.floor(phi/2))
                for omega in range(2**(n-i)):
                    r1_pos = phi + omega*(2**i)
                    a_pos  = psi + 2*omega*(2**(i-1))
                    b_pos  = psi + (2*omega+1)*(2**(i-1))
                    if phi % 2 == 0:
                        a_e.append(a_pos); b_e.append(b_pos)
                        r1_e.append(r1_pos)
                        r1_ep1.append(min(r1_pos+1, N-1))
                    else:
                        a_o.append(a_pos); b_o.append(b_pos)
                        r1_o.append(r1_pos)
                        r1_om1.append(max(r1_pos-1, 0))
            if a_e:
                self.l2r_ops.append({
                    'wi': wi, 'stage': stage, 'parity': 'even',
                    'a'   : torch.tensor(a_e,    dtype=torch.long),
                    'b'   : torch.tensor(b_e,    dtype=torch.long),
                    'r1'  : torch.tensor(r1_e,   dtype=torch.long),
                    'r1p1': torch.tensor(r1_ep1, dtype=torch.long),
                })
            if a_o:
                self.l2r_ops.append({
                    'wi': wi, 'stage': stage, 'parity': 'odd',
                    'a'   : torch.tensor(a_o,    dtype=torch.long),
                    'b'   : torch.tensor(b_o,    dtype=torch.long),
                    'r1'  : torch.tensor(r1_o,   dtype=torch.long),
                    'r1m1': torch.tensor(r1_om1, dtype=torch.long),
                })

        # Register index tensors as buffers
        for idx, op in enumerate(self.r2l_ops):
            for key in ['a','b','r0','r1']:
                self.register_buffer(f'r2l_{idx}_{key}', op[key])
        for idx, op in enumerate(self.l2r_ops):
            self.register_buffer(f'l2r_{idx}_a',  op['a'])
            self.register_buffer(f'l2r_{idx}_b',  op['b'])
            self.register_buffer(f'l2r_{idx}_r1', op['r1'])
            if op['parity'] == 'even':
                self.register_buffer(f'l2r_{idx}_r1p1', op['r1p1'])
            else:
                self.register_buffer(f'l2r_{idx}_r1m1', op['r1m1'])

    @staticmethod
    def f_func(a, b):
        # Smooth atanh approximation — differentiable everywhere
        return 2.0 * torch.atanh(
            torch.clamp(
                torch.tanh(a/2.0) * torch.tanh(b/2.0),
                min=-1+1e-7, max=1-1e-7)
        )

    def forward(self, x_input):
        dev   = x_input.device
        batch = x_input.shape[0]
        x     = torch.clamp(x_input, -10.0, 10.0)

        L = torch.zeros(self.n+1, self.N, batch, device=dev)
        R = torch.zeros(self.n+1, self.N, batch, device=dev)
        L[self.n] = x.T

        for k in range(self.bp_iter_num):
            itr = 0 if self.rnn_share else k

            # Reinit frozen R at stage 0
            R[0, self.frozen_pos, :] = self.inf_num

            # ── R2L pass ─────────────────────────────────────
            for idx, op in enumerate(self.r2l_ops):
                stage = op['stage']
                wi    = op['wi']
                a     = getattr(self, f'r2l_{idx}_a')
                b     = getattr(self, f'r2l_{idx}_b')
                r0    = getattr(self, f'r2l_{idx}_r0')
                r1    = getattr(self, f'r2l_{idx}_r1')

                Lb  = L[stage][b]
                La  = L[stage][a]
                Ri  = R[wi][r1]
                Ri0 = R[wi][r0]

                wRa = self.RV[wi, a, itr].unsqueeze(1)
                wRb = self.RV[wi, b, itr].unsqueeze(1)

                R[stage][a] = wRa * self.f_func(Lb + Ri, Ri0)
                R[stage][b] = Ri  + wRb * self.f_func(La,  Ri0)

            # ── L2R pass ─────────────────────────────────────
            for idx, op in enumerate(self.l2r_ops):
                wi    = op['wi']
                stage = op['stage']
                a     = getattr(self, f'l2r_{idx}_a')
                b     = getattr(self, f'l2r_{idx}_b')
                r1    = getattr(self, f'l2r_{idx}_r1')

                La = L[stage][a]
                Lb = L[stage][b]
                wL = self.LV[wi, r1, itr].unsqueeze(1)

                if op['parity'] == 'even':
                    r1p1  = getattr(self, f'l2r_{idx}_r1p1')
                    Rn    = R[wi][r1p1]
                    new_L = wL * self.f_func(La, Lb + Rn)
                else:
                    r1m1  = getattr(self, f'l2r_{idx}_r1m1')
                    Rp    = R[wi][r1m1]
                    new_L = Lb + wL * self.f_func(La, Rp)

                L[wi][r1] = new_L

            # Clip after each iteration
            L = torch.clamp(L, -50.0, 50.0)
            R = torch.clamp(R, -50.0, 50.0)

        # Output at info positions stage 0
        out = L[0][self.info_pos_t]   # (K, batch)
        out = out.T * -1.0            # (batch, K)
        return out


# ── Build model ───────────────────────────────────────────────
model = NeuralBPDecoderFast(
    N=N, K=K, n=n,
    FZlookup=FZlookup,
    info_pos=info_pos,
    bp_iter_num=bp_iter_num,
    rnn_share=RNN
).to(device)

total_p = sum(p.numel() for p in model.parameters())
print(f'Model   : NeuralBPDecoderFast')
print(f'Device  : {next(model.parameters()).device}')
print(f'Params  : {total_p:,}')
print(f'LV shape: {model.LV.shape}')
print(f'RV shape: {model.RV.shape}')
print(f'R2L ops : {len(model.r2l_ops)}')
print(f'L2R ops : {len(model.l2r_ops)}')

# ── Recreate llr_b for verification ──────────────────────────
ui_b   = np.random.randint(0, 2, (batch_size, K_info))
ucrc_b = np.array([crc_encode(u) for u in ui_b], dtype=np.float32)
UP_b   = np.tile(FZlookup, (batch_size, 1)).copy()
UP_b[UP_b == -1] = ucrc_b.flatten()
UP_b   = UP_b[:, bitreversedindices]
x_b    = np.mod(UP_b @ F_kron_n, 2).astype(np.float32)
llr_b  = (1 - 2*x_b) * 50.0

# Zero-noise BER
with torch.no_grad():
    out_b = model(torch.tensor(llr_b, device=device))
pred_b = (out_b.cpu().numpy() > 0).astype(float)
ber_b  = np.mean(np.abs(ucrc_b - pred_b))
print(f'\nZero-noise BER : {ber_b:.6f}  (must be 0.0)')

# Speed test
t1 = time.time()
for _ in range(10):
    with torch.no_grad():
        _ = model(torch.tensor(llr_b, device=device))
t_per = (time.time()-t1)/10
print(f'Per pass       : {t_per:.3f}s')

# Optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
print(f'\nOptimizer: Adam lr=0.001')
print('Ready for training.')

Model   : NeuralBPDecoderFast
Device  : cuda:0
Params  : 9,216
LV shape: torch.Size([9, 512, 1])
RV shape: torch.Size([9, 512, 1])
R2L ops : 9
L2R ops : 18

Zero-noise BER : 0.000000  (must be 0.0)
Per pass       : 0.078s

Optimizer: Adam lr=0.001
Ready for training.


In [ ]:
# ============================================================
# Training loop — smooth f_func + Adam  (patience=5)
# ============================================================
import h5py
import os

loss_fn      = nn.BCEWithLogitsLoss()
best_val_ber = 1.0
Nobetter     = 0
patience     = 5
history      = {'tl':[], 'tb':[], 'tf':[], 'vl':[], 'vb':[]}
ckpt_path    = f'./bp_{CHANNEL_TYPE}_N{N}_K{K}_iter{bp_iter_num}_RNN{int(RNN)}.weights.h5'

print('='*65)
print(f'NEURAL BP TRAINING — {CHANNEL_TYPE}  N={N}  K={K}')
print(f'BP iters={bp_iter_num}  RNN={RNN}  Adam lr=0.001')
print(f'f_func  : smooth atanh approximation')
print(f'patience: {patience}')
print(f'ckpt    : {ckpt_path}')
print('='*65)

def save_weights_h5(model, path, best_val_ber):
    with h5py.File(path, 'w') as f:
        f.create_dataset('LV', data=model.LV.detach().cpu().numpy())
        f.create_dataset('RV', data=model.RV.detach().cpu().numpy())
        f.attrs['N']           = N
        f.attrs['K']           = K
        f.attrs['E']           = E
        f.attrs['bp_iter_num'] = bp_iter_num
        f.attrs['RNN']         = int(RNN)
        f.attrs['best_val_ber']= best_val_ber
    size_kb = os.path.getsize(path) / 1e3
    print(f'  ✓ saved {path}  ({size_kb:.1f} KB)')

def load_weights_h5(model, path):
    with h5py.File(path, 'r') as f:
        model.LV.data = torch.tensor(
            f['LV'][:], dtype=torch.float32).to(device)
        model.RV.data = torch.tensor(
            f['RV'][:], dtype=torch.float32).to(device)
        saved_ber = f.attrs['best_val_ber']
    print(f'Loaded {path}  best_val_ber={saved_ber:.7f}')
    return model

start = time.time()

for itr in range(50):
    model.train()
    train_nfails = 0
    train_loss   = 0.0
    train_nframe = 0

    # ── Train ────────────────────────────────────────────────
    for i in range(batches_train):
        x_train, y_train = gendata(0, False)
        x_t = torch.tensor(x_train, device=device)
        y_t = torch.tensor(y_train, device=device)

        optimizer.zero_grad()
        yp   = model(x_t)
        loss = loss_fn(yp, y_t)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        train_loss   += loss.item()
        uhat          = (yp.detach().cpu().numpy() >= 0).astype(float)
        train_nfails += np.sum(uhat != y_train)
        train_nframe += np.sum(np.sum(uhat != y_train, axis=1) != 0)

        if (i+1) % 50 == 0:
            elapsed = time.time() - start
            cur_ber = train_nfails / (batch_size * K * (i+1))
            print(f'  Ep {itr+1} batch {i+1}/{batches_train}  '
                  f'loss={train_loss/(i+1):.5f}  '
                  f'BER={cur_ber:.6f}  '
                  f'[{elapsed:.0f}s]')

    # ── Validation ───────────────────────────────────────────
    model.eval()
    val_nfails = 0
    val_loss   = 0.0
    for i in range(batches_val):
        x_val, y_val = gendata(0, False)
        with torch.no_grad():
            x_v  = torch.tensor(x_val, device=device)
            y_v  = torch.tensor(y_val, device=device)
            yp   = model(x_v)
            loss = loss_fn(yp, y_v)
        val_loss   += loss.item()
        uhat        = (yp.cpu().numpy() >= 0).astype(float)
        val_nfails += np.sum(uhat != y_val)

    # ── Epoch summary ────────────────────────────────────────
    t_ber = train_nfails / (batch_size * K * batches_train)
    t_fer = train_nframe / (batch_size * batches_train)
    v_ber = val_nfails   / (batch_size * K * batches_val)
    t_l   = train_loss   / batches_train
    v_l   = val_loss     / batches_val

    history['tl'].append(t_l);  history['tb'].append(t_ber)
    history['tf'].append(t_fer); history['vl'].append(v_l)
    history['vb'].append(v_ber)

    elapsed = time.time() - start
    print(f'Epoch {itr+1:3d}: '
          f'Loss={t_l:.7f}  BER={t_ber:.7f}  FER={t_fer:.7f}  '
          f'Val Loss={v_l:.7f}  Val BER={v_ber:.7f}  '
          f'[{elapsed:.0f}s]', end='')

    # ── Save best ─────────────────────────────────────────────
    if v_ber < best_val_ber:
        best_val_ber = v_ber
        save_weights_h5(model, ckpt_path, best_val_ber)
        Nobetter = 0
    else:
        Nobetter += 1
        print(f'  - {Nobetter}/{patience}')
        if Nobetter in [3, 4]:
            cur_lr = optimizer.param_groups[0]['lr']
            for pg in optimizer.param_groups:
                pg['lr'] = cur_lr * 0.5
            print(f'      LR → {cur_lr*0.5:.7f}')
        if Nobetter > patience:
            print('Early stopping.'); break

    # ── Weight stats every 5 epochs ──────────────────────────
    if (itr+1) % 5 == 0:
        with torch.no_grad():
            print(f'  LV: mean={model.LV.mean():.4f}  '
                  f'std={model.LV.std():.4f}  '
                  f'min={model.LV.min():.4f}  '
                  f'max={model.LV.max():.4f}')
            print(f'  RV: mean={model.RV.mean():.4f}  '
                  f'std={model.RV.std():.4f}  '
                  f'min={model.RV.min():.4f}  '
                  f'max={model.RV.max():.4f}')

print(f'\nDone {(time.time()-start)/60:.1f}min | '
      f'Best Val BER={best_val_ber:.7f}')
print(f'Weights saved → {ckpt_path}')

In [11]:
import torch
import numpy as np
import os
import h5py

# ── Custom loader from your training code ──
def load_weights_h5(model, path):
    with h5py.File(path, 'r') as f:
        model.LV.data = torch.tensor(f['LV'][:], dtype=torch.float32).to(device)
        model.RV.data = torch.tensor(f['RV'][:], dtype=torch.float32).to(device)
        saved_ber = f.attrs['best_val_ber']
    print(f'Loaded {path}  best_val_ber={saved_ber:.7f}')
    return model

# ============================================================
# Test — Neural BP vs SNR
# ============================================================
print('Loading best weights...')
model = load_weights_h5(model, 'bp_PBCH_N512_K56_iter5_RNN1.weights.h5')
model.eval()

print('='*65)
print(f'TEST: Neural BP  [{CHANNEL_TYPE}  N={N}  K={K}  E={E}]')
print(f'BP iters={bp_iter_num}  RNN={RNN}')
print('='*65)
print(f'{"Eb/N0":>8}  {"BER":>12}  {"FER(CRC)":>12}')
print('-'*36)

os.makedirs('Results', exist_ok=True)
results  = []
n_test_b = max(1, batches_test // len(ebn0))

for j in range(len(ebn0)):
    be=0; fe=0; bits=0; frames=0

    for _ in range(n_test_b):
        x_test, y_test = gendata(j, True)
        with torch.no_grad():
            yp = model(torch.tensor(x_test, device=device))
        preds = (yp.cpu().numpy() > 0).astype(int)
        truth = y_test.astype(int)

        be     += np.sum(np.abs(truth - preds))
        for i in range(preds.shape[0]):
            fe += 0 if crc_check(preds[i]) else 1
        bits   += truth.size
        frames += truth.shape[0]

    ber_v = be / bits
    fer_v = fe / frames
    results.append((ebn0[j], ber_v, fer_v))
    print(f'{ebn0[j]:>8.1f}  {ber_v:>12.6f}  {fer_v:>12.6f}')

    if fe == 0 and j >= 3:
        print('  (zero frame errors — stopping)')
        break

# Save results
rf = f'Results/bp_PBCH_N512_K56_iter5_RNN1_results.txt'
with open(rf, 'w') as f:
    f.write(f'Neural BP  {CHANNEL_TYPE}  N={N} K={K} E={E} '
            f'iters={bp_iter_num} RNN={RNN}\n')
    f.write(f'{"Eb/N0":>8}  {"BER":>12}  {"FER_CRC":>12}\n')
    for r in results:
        f.write(f'{r[0]:>8.1f}  {r[1]:>12.6e}  {r[2]:>12.6e}\n')

print(f'\nResults saved → {rf}')

Loading best weights...
Loaded bp_PBCH_N512_K56_iter5_RNN1.weights.h5  best_val_ber=0.1872750
TEST: Neural BP  [PBCH  N=512  K=56  E=864]
BP iters=5  RNN=1
   Eb/N0           BER      FER(CRC)
------------------------------------
    -3.0      0.472660      1.000000
    -2.0      0.452499      0.999600
    -1.0      0.408746      0.992500
     0.0      0.314191      0.924083
     1.0      0.175632      0.669167
     2.0      0.058711      0.291450
     3.0      0.011064      0.065967
     4.0      0.001319      0.009267
     5.0      0.000126      0.000900
     6.0      0.000011      0.000167

Results saved → Results/bp_PBCH_N512_K56_iter5_RNN1_results.txt


In [14]:
# ===================================================================
# NEURAL BP DECODER — LIVE DEMO & VERIFICATION (PBCH)
# ===================================================================
import torch
import numpy as np
import h5py   # ← Fixed: Import h5py here
import os

# ── Load best trained weights ─────────────────────────────────────
ckpt_path = f'bp_{CHANNEL_TYPE}_N{N}_K{K}_iter{bp_iter_num}_RNN{int(RNN)}.weights.h5'

def load_weights_h5(model, path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"❌ Weight file not found: {path}")
    
    with h5py.File(path, 'r') as f:
        model.LV.data = torch.tensor(f['LV'][:], dtype=torch.float32).to(device)
        model.RV.data = torch.tensor(f['RV'][:], dtype=torch.float32).to(device)
        saved_ber = f.attrs.get('best_val_ber', 0.0)
    
    print(f"✓ Weights loaded successfully from:\n   {path}")
    return model

# Load the model
model = load_weights_h5(model, ckpt_path)
model.eval()

# ── Demo Parameters ───────────────────────────────────────────────
DEMO_SNR_DB  = 4.0
N_DEMO       = 1000
SHOW_CORRECT = 8
SHOW_WRONG   = 2

si = int(np.argmin(np.abs(ebn0 - DEMO_SNR_DB)))

print("=" * 90)
print(f"  NEURAL BP DECODER LIVE DEMO  |  Eb/N0 = {ebn0[si]:.1f} dB  |  {N_DEMO} samples on-the-fly")
print(f"  BP iterations = {bp_iter_num}   RNN weight sharing = {RNN}")
print(f"  CRC-guided single-bit correction ENABLED")
print("=" * 90)
print(f"  Polar code : N={N}, K={K} (K_info={K_info} + CRC={CRC_LEN})")
print(f"  Rate-match : E={E}  (QPSK → {E//2} symbols)")

print()

def bits_to_str(arr, group=8):
    s = ''.join(str(int(b)) for b in arr)
    return ' '.join(s[i:i+group] for i in range(0, len(s), group))

correct_shown    = 0
wrong_shown      = 0
correct_total    = 0
wrong_total      = 0
corrected_total  = 0
total            = 0

print("─" * 90)
print(f"  Processing {N_DEMO} samples …  (first {SHOW_CORRECT} ✓  +  first {SHOW_WRONG} ✗ shown)")
print("─" * 90 + "\n")

# ── CRC single-bit correction helper ───────────────────────────────
def crc_correct(bits, logits):
    """Try flipping the least confident bits until CRC passes"""
    if crc_check(bits.astype(int)):
        return bits.copy(), 0
    confidence = np.abs(logits)
    order = np.argsort(confidence)          # least confident first
    corrected = bits.copy()
    for idx in order:
        corrected[idx] ^= 1
        if crc_check(corrected.astype(int)):
            return corrected, int(np.sum(corrected != bits))
        corrected[idx] ^= 1
    return bits.copy(), 0

def print_block(label, uc, x, x_dec, nsc_pred, logits, n_bit_errors, crc_ok, flips=0, err_pos=None):
    tag   = "✓ CORRECT" if n_bit_errors == 0 else f"✗ WRONG  ({n_bit_errors} bit-error{'s' if n_bit_errors>1 else ''})"
    crc_s = "PASS" if crc_ok else "FAIL"
    flip_s = f"  [CRC corrected {flips} bit{'s' if flips>1 else ''}]" if flips > 0 else ""
    
    print(f"  [{label}]  {tag}  |  CRC-24C: {crc_s}{flip_s}")
    
    print(f"  ┌─ TX INFO (32 bit) : {bits_to_str(uc[:32])}")
    print(f"  │  TX CRC  (24 bit) : {bits_to_str(uc[32:])}")
    print(f"  │")
    
    # Coded word
    coded_str = ''.join(str(int(b)) for b in x)
    print(f"  │  512-BIT CODED WORD (TX polar codeword, 8×64 bits):")
    for row in range(8):
        seg = coded_str[row*64:(row+1)*64]
        print(f"  │    [{row*64:03d}–{row*64+63:03d}]  {' '.join(seg[i:i+8] for i in range(0,64,8))}")
    print(f"  │")
    
    # Decoded word
    dec_str = ''.join(str(int(b)) for b in x_dec)
    print(f"  │  512-BIT DECODED WORD (from Neural BP):")
    for row in range(8):
        seg = dec_str[row*64:(row+1)*64]
        print(f"  │    [{row*64:03d}–{row*64+63:03d}]  {' '.join(seg[i:i+8] for i in range(0,64,8))}")
    print(f"  │")
    
    # BP Output - clear split
    msg_bits = nsc_pred[:32]
    crc_bits = nsc_pred[32:]
    print(f"  │  NEURAL BP OUTPUT (K=56 bits) :")
    print(f"  │     Message (32 bits) → {bits_to_str(msg_bits)}")
    print(f"  │     CRC-24C (24 bits) → {bits_to_str(crc_bits)}")
    
    # LLR statistics
    print(f"  │")
    print(f"  │  LLRs fed to Neural BP Decoder (N={N}):")
    print(f"  │     Min   = {logits.min():.3f}   Max   = {logits.max():.3f}")
    print(f"  │     Mean  = {logits.mean():.3f}   Std   = {logits.std():.3f}")
    print(f"  │     First 32 LLRs : {np.round(logits[:32], 3)}")
    
    if err_pos is not None and len(err_pos) > 0:
        print(f"  │")
        print(f"  │  TRUE    (K=56 bits)    : {bits_to_str(uc)}")
        err_line = ' '.join(''.join('^' if i+g in set(err_pos) else ' ' for i in range(8)) for g in range(0,K,8))
        print(f"  │  ERROR POSITIONS        : {err_line}")
        print(f"  │  → Error bit indices    : {list(err_pos)}")
    
    print(f"  └{'─'*82}\n")

# ── Main Demo Loop ─────────────────────────────────────────────────
for sample_idx in range(N_DEMO):
    ui  = np.random.randint(0, 2, K_info)
    uc  = crc_encode(ui)

    # Polar Encoding
    UP = np.tile(FZlookup, (1, 1)).copy()
    UP[0][UP[0] == -1] = uc
    UP = UP[:, bitreversedindices]
    x  = np.mod(UP @ F_kron_n, 2).astype(np.float32)[0]

    # Rate matching + Channel
    repeat = int(np.ceil(E / N))
    x_rep  = np.tile(x, repeat)[:E]
    xp     = x_rep.reshape(E//2, 2)
    tx     = ((1 - 2*xp[:,0]) + 1j*(1 - 2*xp[:,1])) / np.sqrt(2)

    s2     = noise_var[si]
    rx     = tx + (np.random.normal(0, np.sqrt(s2), tx.shape) + 
                   1j*np.random.normal(0, np.sqrt(s2), tx.shape))

    scale  = 2.0 / s2
    llr_E  = np.stack([scale * np.real(rx), scale * np.imag(rx)], axis=1).reshape(E)

    llr_N = np.zeros(N, dtype=np.float32)
    for k in range(repeat):
        st = k * N
        en = min(st + N, E)
        llr_N[:en-st] += llr_E[st:en]

    # Neural BP Decoding
    with torch.no_grad():
        logits_t = model(torch.tensor(llr_N.reshape(1, N), device=device))
        logits   = logits_t.cpu().numpy()[0]

    nsc_pred = (logits > 0).astype(int)

    # CRC correction
    nsc_pred, flips = crc_correct(nsc_pred, logits)
    if flips > 0:
        corrected_total += 1

    # Reconstruct codeword for display
    UP_dec = np.tile(FZlookup, (1, 1)).copy()
    UP_dec[0][UP_dec[0] == -1] = nsc_pred
    UP_dec = UP_dec[:, bitreversedindices]
    x_dec  = np.mod(UP_dec @ F_kron_n, 2).astype(int)[0]

    n_bit_errors = int(np.sum(np.abs(uc - nsc_pred)))
    crc_ok       = crc_check(nsc_pred)
    is_correct   = (n_bit_errors == 0)

    total += 1
    if is_correct:
        correct_total += 1
    else:
        wrong_total += 1

    if is_correct and correct_shown < SHOW_CORRECT:
        print_block(f"CORRECT #{correct_shown+1}", uc, x, x_dec, nsc_pred, logits,
                    n_bit_errors, crc_ok, flips)
        correct_shown += 1
    elif not is_correct and wrong_shown < SHOW_WRONG:
        err_pos = np.where(np.abs(uc - nsc_pred))[0]
        print_block(f"WRONG   #{wrong_shown+1}", uc, x, x_dec, nsc_pred, logits,
                    n_bit_errors, crc_ok, flips, err_pos)
        wrong_shown += 1

# ── Final Summary ─────────────────────────────────────────────────
val_prob = correct_total / total
fer_val  = wrong_total / total

print("=" * 90)
print(f"          {N_DEMO} on-the-fly samples ")
print("=" * 90)
print(f"\n  Total samples processed   : {total:>6d}")
print(f"  ✓ Correct words           : {correct_total:>6d}")
print(f"  ✗ Wrong words             : {wrong_total:>6d}")
print(f"  🔧 CRC-corrected words    : {corrected_total:>6d}  (single-bit flips that passed CRC)")
print()
print(f"  ┌─────────────────────────────────────────────────────────────┐")
print(f"  │  P_correct = {val_prob:.6f}   ({val_prob*100:.3f}% words decoded correctly)       │")
print(f"  │  FER       = {fer_val:.6f}   ({fer_val*100:.3f}% words had ≥1 bit error)          │")
print(f"  └─────────────────────────────────────────────────────────────┘")
print("=" * 90)

✓ Weights loaded successfully from:
   bp_PBCH_N512_K56_iter5_RNN1.weights.h5
  NEURAL BP DECODER LIVE DEMO  |  Eb/N0 = 4.0 dB  |  1000 samples on-the-fly
  BP iterations = 5   RNN weight sharing = 1
  CRC-guided single-bit correction ENABLED
  Polar code : N=512, K=56 (K_info=32 + CRC=24)
  Rate-match : E=864  (QPSK → 432 symbols)

──────────────────────────────────────────────────────────────────────────────────────────
  Processing 1000 samples …  (first 8 ✓  +  first 2 ✗ shown)
──────────────────────────────────────────────────────────────────────────────────────────

  [CORRECT #1]  ✓ CORRECT  |  CRC-24C: PASS
  ┌─ TX INFO (32 bit) : 11101011 11110100 00100010 00010110
  │  TX CRC  (24 bit) : 11101110 01111101 01111000
  │
  │  512-BIT CODED WORD (TX polar codeword, 8×64 bits):
  │    [000–063]  00000000 11000011 10011001 01011010 01100110 01011010 00000000 00111100
  │    [064–127]  10011001 01011010 11111111 00111100 11111111 11000011 10011001 10100101
  │    [128–191]  00110011